## 1. Setup & Data

In [1]:
 # ============================================================
# V2 Inventory Simulation
# ============================================================

import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from src.inventory.policy import (
    calculate_target_inventory,
    calculate_recommended_order_qty,
)

from src.inventory.reorder import (
    calculate_reorder_point,
    should_reorder,
)

from src.inventory.safety_stock import (
    calculate_safety_stock,
)

from src.inventory.simulation import (
    run_backtest,
    run_baseline_backtest,
    get_arrivals_for_date,
    process_daily_demand,
    create_purchase_order,
    calculate_inventory_position,
)

from src.models.baselines import moving_average_forecast
from src.evaluation.metrics import calculate_inventory_metrics
from src.inventory.config import prepare_product_inventory_config


In [3]:
sales = pd.read_csv(
    PROJECT_ROOT / "data" / "raw" / "sales.csv"
)

sales["date"] = pd.to_datetime(sales["date"])


print("Sales shape:", sales.shape)
print("Sales date range:", sales["date"].min(), "to", sales["date"].max())
print("Products:", sales["product_id"].unique())

Sales shape: (3655, 11)
Sales date range: 2024-01-01 00:00:00 to 2025-12-31 00:00:00
Products: <StringArray>
['P001', 'P002', 'P003', 'P004', 'P005']
Length: 5, dtype: str


In [4]:
product_ids = sorted(sales["product_id"].unique())

print("Products:", product_ids)
print("Number of products:", len(product_ids))

Products: ['P001', 'P002', 'P003', 'P004', 'P005']
Number of products: 5


In [5]:
MODEL_PATH = (
    PROJECT_ROOT
    / "models"
    / "xgboost_forecaster.joblib"
)

model = joblib.load(MODEL_PATH)

MODEL_FEATURES = [
    "price",
    "discount",
    "promotion",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_28",
    "rolling_std_7",
]

print("Model loaded:", type(model))

Model loaded: <class 'xgboost.sklearn.XGBRegressor'>


In [6]:
backtest_start = pd.Timestamp("2025-10-01")
backtest_end = pd.Timestamp("2025-12-31")

lead_time_days = 4
inventory_days = 5

print("Backtest:", backtest_start.date(), "to", backtest_end.date())
print("Lead time:", lead_time_days, "days")
print("Inventory days:", inventory_days)

Backtest: 2025-10-01 to 2025-12-31
Lead time: 4 days
Inventory days: 5


In [7]:
holding_cost_rate = 0.20
ordering_cost_per_order = 500
stockout_cost_per_unit = 1000
unit_cost = 1000

backtest_days_count = (
    backtest_end - backtest_start
).days + 1

In [8]:
forecast_error_std = pd.read_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "forecast_error_std.csv"
)

   

In [9]:
all_product_results = []

for product_id in product_ids:
    print(f"Running V2 simulation for {product_id}...")

    product_history = (
        sales[sales["product_id"] == product_id]
        .copy()
        .sort_values("date")
        .reset_index(drop=True)
    )

    product_config = prepare_product_inventory_config(
        product_id=product_id,
        product_history=product_history,
        forecast_error_std=forecast_error_std,
        backtest_start=backtest_start,
        lead_time_days=lead_time_days,
        inventory_days=inventory_days,
    )

    xgb_results = run_backtest(
        product_history=product_history,
        start_date=backtest_start,
        end_date=backtest_end,
        starting_stock=product_config["starting_stock"],
        model=model,
        model_features=MODEL_FEATURES,
        safety_stock=product_config["safety_stock"],
        lead_time_days=product_config["lead_time_days"],
    )

    baseline_results = run_baseline_backtest(
        product_history=product_history,
        start_date=backtest_start,
        end_date=backtest_end,
        starting_stock=product_config["starting_stock"],
        safety_stock=product_config["safety_stock"],
        lead_time_days=product_config["lead_time_days"],
    )

    xgb_metrics = calculate_inventory_metrics(xgb_results)
    baseline_metrics = calculate_inventory_metrics(baseline_results)

    all_product_results.append(
        {
            "product_id": product_id,
            "xgb_metrics": xgb_metrics,
            "baseline_metrics": baseline_metrics,
        }
    )

print("Completed:", len(all_product_results), "products")

Running V2 simulation for P001...


Running V2 simulation for P002...
Running V2 simulation for P003...
Running V2 simulation for P004...
Running V2 simulation for P005...
Completed: 5 products


In [10]:
product_comparison = []

for result in all_product_results:
    product_id = result["product_id"]
    xgb = result["xgb_metrics"]
    baseline = result["baseline_metrics"]

    product_comparison.append(
        {
            "Product": product_id,
            "XGBoost Avg Inventory": xgb["average_inventory"],
            "Moving Average Avg Inventory": baseline["average_inventory"],
            "XGBoost Stockout Days": xgb["stockout_days"],
            "Moving Average Stockout Days": baseline["stockout_days"],
            "XGBoost Lost Sales": xgb["lost_sales_units"],
            "Moving Average Lost Sales": baseline["lost_sales_units"],
            "XGBoost Service Level (%)": xgb["service_level"],
            "Moving Average Service Level (%)": baseline["service_level"],
            "XGBoost Orders": xgb["number_of_orders"],
            "Moving Average Orders": baseline["number_of_orders"],
        }
    )

product_comparison = pd.DataFrame(product_comparison)

product_comparison

,Product,XGBoost Avg Inventory,Moving Average Avg Inventory,XGBoost Stockout Days,Moving Average Stockout Days,XGBoost Lost Sales,Moving Average Lost Sales,XGBoost Service Level (%),Moving Average Service Level (%),XGBoost Orders,Moving Average Orders
0,P001,374.923913,544.413043,2,0,26,0,99.329724,100.000000,5,4
1,P002,293.586957,436.184783,2,0,28,0,98.969452,100.000000,5,4
2,P003,341.554348,485.836957,1,0,12,0,99.630656,100.000000,5,4
3,P004,251.804348,352.369565,1,1,12,12,99.447514,99.447514,5,4
4,P005,483.641304,655.760870,0,0,0,0,100.000000,100.000000,5,4


In [11]:
cost_comparison = []

for result in all_product_results:
    product_id = result["product_id"]

    xgb = result["xgb_metrics"]
    baseline = result["baseline_metrics"]

    xgb_holding_cost = (
        xgb["average_inventory"]
        * unit_cost
        * holding_cost_rate
        * backtest_days_count / 365
    )

    xgb_ordering_cost = (
        xgb["number_of_orders"]
        * ordering_cost_per_order
    )

    xgb_stockout_cost = (
        xgb["lost_sales_units"]
        * stockout_cost_per_unit
    )

    xgb_total_cost = (
        xgb_holding_cost
        + xgb_ordering_cost
        + xgb_stockout_cost
    )

    baseline_holding_cost = (
        baseline["average_inventory"]
        * unit_cost
        * holding_cost_rate
        * backtest_days_count / 365
    )

    baseline_ordering_cost = (
        baseline["number_of_orders"]
        * ordering_cost_per_order
    )

    baseline_stockout_cost = (
        baseline["lost_sales_units"]
        * stockout_cost_per_unit
    )

    baseline_total_cost = (
        baseline_holding_cost
        + baseline_ordering_cost
        + baseline_stockout_cost
    )

    cost_comparison.append(
        {
            "Product": product_id,
            "XGBoost Total Cost (₹)": xgb_total_cost,
            "Moving Average Total Cost (₹)": baseline_total_cost,
            "Savings with XGBoost (₹)": (
                baseline_total_cost - xgb_total_cost
            ),
        }
    )

cost_comparison = pd.DataFrame(cost_comparison)

cost_comparison

,Product,XGBoost Total Cost (₹),Moving Average Total Cost (₹),Savings with XGBoost (₹)
0,P001,47400.273973,29444.383562,-17955.890411
1,P002,45300.000000,23988.493151,-21311.506849
2,P003,31718.082192,26491.506849,-5226.575342
3,P004,27193.698630,31763.287671,4569.589041
4,P005,26880.821918,35057.534247,8176.712329


In [12]:
overall_cost_comparison = pd.DataFrame(
    {
        "Strategy": [
            "XGBoost",
            "Moving Average",
        ],
        "Total Inventory Cost (₹)": [
            cost_comparison["XGBoost Total Cost (₹)"].sum(),
            cost_comparison["Moving Average Total Cost (₹)"].sum(),
        ],
    }
)

overall_cost_comparison["Difference vs XGBoost (₹)"] = (
    overall_cost_comparison["Total Inventory Cost (₹)"]
    - overall_cost_comparison.loc[
        overall_cost_comparison["Strategy"] == "XGBoost",
        "Total Inventory Cost (₹)"
    ].iloc[0]
)

overall_cost_comparison

,Strategy,Total Inventory Cost (₹),Difference vs XGBoost (₹)
0,XGBoost,178492.876712,0.000000
1,Moving Average,146745.205479,-31747.671233


In [ ]:
overall_xgb = {}
overall_baseline = {}

for result in all_product_results:
    xgb = result["xgb_metrics"]
    baseline = result["baseline_metrics"]

    for key, value in xgb.items():
        if key not in ["average_inventory", "maximum_inventory"]:
            overall_xgb[key] = (
                overall_xgb.get(key, 0) + value
            )

    for key, value in baseline.items():
        if key not in ["average_inventory", "maximum_inventory"]:
            overall_baseline[key] = (
                overall_baseline.get(key, 0) + value
            )

overall_xgb_service_level = (
    overall_xgb["total_fulfilled"]
    / overall_xgb["total_demand"]
    * 100
)

overall_baseline_service_level = (
    overall_baseline["total_fulfilled"]
    / overall_baseline["total_demand"]
    * 100
)

overall_metrics = pd.DataFrame(
    {
        "Metric": [
            "Total Units Ordered",
            "Number of Orders",
            "Stockout Days",
            "Lost Sales Units",
            "Total Demand",
            "Total Fulfilled",
            "Service Level (%)",
        ],
        "XGBoost": [
            overall_xgb["total_units_ordered"],
            overall_xgb["number_of_orders"],
            overall_xgb["stockout_days"],
            overall_xgb["lost_sales_units"],
            overall_xgb["total_demand"],
            overall_xgb["total_fulfilled"],
            overall_xgb_service_level,
        ],
        "Moving Average": [
            overall_baseline["total_units_ordered"],
            overall_baseline["number_of_orders"],
            overall_baseline["stockout_days"],
            overall_baseline["lost_sales_units"],
            overall_baseline["total_demand"],
            overall_baseline["total_fulfilled"],
            overall_baseline_service_level,
        ],
    }
)

overall_metrics

,Metric,XGBoost,Moving Average
0,Average Inventory,1745.510870,2474.565217
1,Maximum Inventory,3818.000000,5555.000000
2,Total Units Ordered,18123.000000,18555.000000
3,Number of Orders,25.000000,20.000000
4,Stockout Days,6.000000,1.000000
5,Lost Sales Units,78.000000,12.000000
6,Service Level (%),99.519852,99.926131


In [14]:
policy_recommendations = cost_comparison.copy()

policy_recommendations["Recommended Strategy"] = policy_recommendations.apply(
    lambda row: (
        "XGBoost"
        if row["XGBoost Total Cost (₹)"] < row["Moving Average Total Cost (₹)"]
        else "Moving Average"
    ),
    axis=1,
)

policy_recommendations["Expected Savings (₹)"] = (
    policy_recommendations[
        ["XGBoost Total Cost (₹)", "Moving Average Total Cost (₹)"]
    ].max(axis=1)
    - policy_recommendations[
        ["XGBoost Total Cost (₹)", "Moving Average Total Cost (₹)"]
    ].min(axis=1)
)

policy_recommendations[
    [
        "Product",
        "Recommended Strategy",
        "Expected Savings (₹)",
    ]
]

,Product,Recommended Strategy,Expected Savings (₹)
0,P001,Moving Average,17955.890411
1,P002,Moving Average,21311.506849
2,P003,Moving Average,5226.575342
3,P004,XGBoost,4569.589041
4,P005,XGBoost,8176.712329


In [15]:
stockout_cost_scenarios = [500, 1000, 2000, 5000]

sensitivity_results = []

for stockout_cost in stockout_cost_scenarios:
    xgb_total = 0
    baseline_total = 0

    for result in all_product_results:
        xgb = result["xgb_metrics"]
        baseline = result["baseline_metrics"]

        xgb_holding = (
            xgb["average_inventory"]
            * unit_cost
            * holding_cost_rate
            * backtest_days_count / 365
        )

        xgb_ordering = (
            xgb["number_of_orders"]
            * ordering_cost_per_order
        )

        xgb_stockout = (
            xgb["lost_sales_units"]
            * stockout_cost
        )

        baseline_holding = (
            baseline["average_inventory"]
            * unit_cost
            * holding_cost_rate
            * backtest_days_count / 365
        )

        baseline_ordering = (
            baseline["number_of_orders"]
            * ordering_cost_per_order
        )

        baseline_stockout = (
            baseline["lost_sales_units"]
            * stockout_cost
        )

        xgb_total += (
            xgb_holding
            + xgb_ordering
            + xgb_stockout
        )

        baseline_total += (
            baseline_holding
            + baseline_ordering
            + baseline_stockout
        )

    sensitivity_results.append(
        {
            "Stockout Cost (₹/unit)": stockout_cost,
            "XGBoost Total Cost (₹)": xgb_total,
            "Moving Average Total Cost (₹)": baseline_total,
            "Cheaper Strategy": (
                "XGBoost"
                if xgb_total < baseline_total
                else "Moving Average"
            ),
        }
    )

sensitivity_results = pd.DataFrame(sensitivity_results)

sensitivity_results

,Stockout Cost (₹/unit),XGBoost Total Cost (₹),Moving Average Total Cost (₹),Cheaper Strategy
0,500,139492.876712,140745.205479,XGBoost
1,1000,178492.876712,146745.205479,Moving Average
2,2000,256492.876712,158745.205479,Moving Average
3,5000,490492.876712,194745.205479,Moving Average


In [16]:
cost_at_500 = sensitivity_results.loc[
    sensitivity_results["Stockout Cost (₹/unit)"] == 500
].iloc[0]

cost_at_1000 = sensitivity_results.loc[
    sensitivity_results["Stockout Cost (₹/unit)"] == 1000
].iloc[0]

difference_at_500 = (
    cost_at_500["XGBoost Total Cost (₹)"]
    - cost_at_500["Moving Average Total Cost (₹)"]
)

difference_at_1000 = (
    cost_at_1000["XGBoost Total Cost (₹)"]
    - cost_at_1000["Moving Average Total Cost (₹)"]
)

break_even_stockout_cost = (
    500
    + (
        -difference_at_500
        * (1000 - 500)
        / (difference_at_1000 - difference_at_500)
    )
)

print(
    f"Break-even stockout cost: "
    f"₹{break_even_stockout_cost:,.2f} per unit"
)

Break-even stockout cost: ₹518.97 per unit


## V2 Conclusion

### What V2 Evaluated

V2 replayed the inventory decision process across all five products from **2025-10-01 to 2025-12-31**.

Two forecasting strategies were evaluated using the same inventory policy:

- XGBoost
- 7-day Moving Average

### Key Findings

- XGBoost consistently maintained lower average inventory than the Moving Average baseline.
- Moving Average generally achieved a higher service level with fewer stockouts.
- The economically preferred forecasting strategy differs by product.
- Moving Average was the cheaper strategy for P001, P002, and P003.
- XGBoost was the cheaper strategy for P004 and P005.
- Across all five products, Moving Average had the lower total inventory cost under the default cost assumptions.
- Sensitivity analysis showed that the preferred strategy changes when the cost of stockouts changes.
- The break-even stockout cost was approximately ₹518.97 per unit.

### Business Decision

V2 demonstrates that the best forecasting strategy cannot be selected using forecast accuracy alone.

The forecasting model must be evaluated through its downstream inventory decisions, service-level impact, stockout risk, and total operating cost.

Therefore, V2 transforms the system from:

**"Which model predicts demand better?"**

into:

**"Which forecasting strategy makes better inventory decisions for the business?"**